In [16]:
import numpy as np
import pandas as pd
import datetime
import yfinance as yf
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

# For importing universal scripts
import sys
import os
# Go up two levels from the subfolder
sys.path.append(os.path.abspath(".."))
from indicators_returns import final_df #Universal script for indicator set and actuals
import importlib
import indicators_returns
importlib.reload(indicators_returns)
from indicators_returns import final_df
import gc
from sklearn.metrics import (fbeta_score, accuracy_score, f1_score, 
                             confusion_matrix, balanced_accuracy_score, recall_score, matthews_corrcoef, precision_score)
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split, StratifiedKFold
from xgboost import XGBClassifier
import math
import pickle
from sklearn.inspection import permutation_importance
from sklearn.model_selection import TimeSeriesSplit

tags = pd.read_csv('../Indicator_Selection_Pipeline/Finalization/tags_cons.csv') 

def extract(ticker, returns, lb, cat_cols_all, windows=[10, 25]):

    import os
    # point this at an existing directory in your project
    os.chdir('/Users/brettchase/Documents/Fracturion/Feature_Set_Construction_Testing')
    
    df = final_df(ticker, returns, lb)
    df = df.iloc[:-101].replace([np.inf, -np.inf], 0)#

    df = df.sort_index(ascending=True)
    # Exponential Moving Average
    ema_cols = {
        f"{col}_EMA{w}": df[col].ewm(span=w, adjust=False).mean()
        for w in windows
        for col in cat_cols_all
    }
    # 3) merge them back into one dict
    new_cols = {**ema_cols}

    # 4) concatenate onto your original df
    df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)    
    df = df.sort_index(ascending=False)
    
    return df

lb = 8
returns = [5, 10, 15, 20, 25, 30]
raw_all = tags['Indicator'][tags['Type'] == 'Raw']
df = extract('QQQ', returns, lb, raw_all, windows=[10,25])

# By category × velocity
duration_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
duration_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
duration_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'duration') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
trend_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
trend_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
trend_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
trend_ratio_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
trend_ratio_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
trend_ratio_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'trend_ratio') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
volatility_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
volatility_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
volatility_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'volatility') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
momentum_slow = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'slow')]['Indicator'].tolist()
momentum_moderate = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'moderate')]['Indicator'].tolist()
momentum_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()
lag_fast = tags[(tags['Type'] == 'Raw') & (tags['Category'] == 'momentum') & (tags['velocity_25_cat_local'] == 'fast')]['Indicator'].tolist()

# Start with must include columns (slope)

In [17]:
def add_cyclic_seasonality(
    df: pd.DataFrame,
    date_col: str = 'Date',
    add_weekly: bool = True,
    add_month: bool = True,
    add_quarter: bool = True,
    add_year: bool = True,
    add_day_of_year: bool = False,
    mode: str = 'calendar',          # 'trading', 'calendar', or 'both'
    prefix: str = 'cyc_'
) -> pd.DataFrame:
    """
    Add cyclic (sin/cos) seasonality features.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe containing a date column.
    date_col : str
        Name of the datetime column.
    add_weekly, add_month, add_quarter, add_year, add_day_of_year : bool
        Toggles for which seasonal cycles to create.
    mode : {'trading','calendar','both'}
        - 'trading': position = trading-day index / trading-days-in-period (resets at dataset slice start)
        - 'calendar': position = calendar progress within period (day-based; retains true mid-period phase)
        - 'both': create both sets (with suffixes _trade and _cal)
    prefix : str
        Prefix for created feature names.

    Returns
    -------
    pd.DataFrame
        Copy of df with new seasonality features appended (original row order preserved).
    """
    if date_col not in df.columns:
        raise ValueError(f"'{date_col}' not found in DataFrame.")
    if mode not in {'trading', 'calendar', 'both'}:
        raise ValueError("mode must be 'trading', 'calendar', or 'both'.")

    out = df.copy()
    out[date_col] = pd.to_datetime(out[date_col])

    # Preserve original order / index
    original_index = out.index
    work = out.sort_values(date_col).reset_index(drop=False)
    idx_col = work.columns[0]

    # Precompute basic date parts
    work['_year']    = work[date_col].dt.year
    work['_month']   = work[date_col].dt.month
    work['_quarter'] = work[date_col].dt.quarter
    work['_weekday'] = work[date_col].dt.weekday
    work['_doy']     = work[date_col].dt.dayofyear
    work['_is_leap'] = work[date_col].dt.is_leap_year

    # ---------- WEEKLY (trading week: Mon-Fri) ----------
    if add_weekly:
        week_pos = work['_weekday'] / 5.0  # Monday=0.0, Friday≈0.8 (if only Mon-Fri present)
        work[f'{prefix}week_sin'] = np.sin(2 * np.pi * week_pos)
        work[f'{prefix}week_cos'] = np.cos(2 * np.pi * week_pos)

    # Helper to safely divide
    def _safe_div(num, den):
        return np.where(den == 0, 0.0, num / np.where(den == 0, 1, den))

    # ---------- MONTH ----------
    if add_month:
        if mode in {'trading', 'both'}:
            grp_m = [work['_year'], work['_month']]
            idx_m = work.groupby(grp_m).cumcount()
            cnt_m = work.groupby(grp_m)[date_col].transform('count')
            pos_m_trade = _safe_div(idx_m, cnt_m)
            work[f'{prefix}month_trade_sin'] = np.sin(2 * np.pi * pos_m_trade)
            work[f'{prefix}month_trade_cos'] = np.cos(2 * np.pi * pos_m_trade)

        if mode in {'calendar', 'both'}:
            dom = work[date_col].dt.day           # 1..days_in_month
            dim = work[date_col].dt.days_in_month
            pos_m_cal = _safe_div(dom - 1, dim - 1)
            work[f'{prefix}month_sin'] = np.sin(2 * np.pi * pos_m_cal)
            work[f'{prefix}month_cos'] = np.cos(2 * np.pi * pos_m_cal)

    # ---------- QUARTER ----------
    if add_quarter:
        if mode in {'trading', 'both'}:
            grp_q = [work['_year'], work['_quarter']]
            idx_q = work.groupby(grp_q).cumcount()
            cnt_q = work.groupby(grp_q)[date_col].transform('count')
            pos_q_trade = _safe_div(idx_q, cnt_q)
            work[f'{prefix}quarter_trade_sin'] = np.sin(2 * np.pi * pos_q_trade)
            work[f'{prefix}quarter_trade_cos'] = np.cos(2 * np.pi * pos_q_trade)

        if mode in {'calendar', 'both'}:
            q = work['_quarter']
            y = work['_year']
            quarter_start = pd.to_datetime({'year': y, 'month': (q - 1) * 3 + 1, 'day': 1})
            quarter_end = quarter_start + pd.offsets.QuarterEnd(0)
            day_in_q = (work[date_col] - quarter_start).dt.days
            days_q = (quarter_end - quarter_start).dt.days
            pos_q_cal = _safe_div(day_in_q, days_q)
            work[f'{prefix}quarter_sin'] = np.sin(2 * np.pi * pos_q_cal)
            work[f'{prefix}quarter_cos'] = np.cos(2 * np.pi * pos_q_cal)

    # ---------- YEAR ----------
    if add_year:
        if mode in {'trading', 'both'}:
            grp_y = work['_year']
            idx_y = work.groupby(grp_y).cumcount()
            cnt_y = work.groupby(grp_y)[date_col].transform('count')
            pos_y_trade = _safe_div(idx_y, cnt_y)
            work[f'{prefix}year_trade_sin'] = np.sin(2 * np.pi * pos_y_trade)
            work[f'{prefix}year_trade_cos'] = np.cos(2 * np.pi * pos_y_trade)

        if mode in {'calendar', 'both'}:
            doy = work['_doy']
            year_len = np.where(work['_is_leap'], 366, 365)
            pos_y_cal = _safe_div(doy - 1, year_len - 1)
            work[f'{prefix}year_sin'] = np.sin(2 * np.pi * pos_y_cal)
            work[f'{prefix}year_cos'] = np.cos(2 * np.pi * pos_y_cal)

    # ---------- DAY OF YEAR (explicit) ----------
    if add_day_of_year:
        year_len = np.where(work['_is_leap'], 366, 365)
        work[f'{prefix}doy_sin'] = np.sin(2 * np.pi * work['_doy'] / year_len)
        work[f'{prefix}doy_cos'] = np.cos(2 * np.pi * work['_doy'] / year_len)

    # Drop temp columns
    temp_cols = [c for c in work.columns if c.startswith('_')]
    work = work.drop(columns=temp_cols)

    # Identify newly created features
    new_cols = [c for c in work.columns if c not in out.columns and c != idx_col]

    # Map back to original order
    work = work.set_index(idx_col)
    out[new_cols] = work.loc[original_index, new_cols]

    return out

weekly = ['cyc_week_sin', 'cyc_week_cos']
monthly = ['cyc_month_sin', 'cyc_month_cos']
quarterly = ['cyc_quarter_sin', 'cyc_quarter_cos']
yearly = ['cyc_year_sin', 'cyc_year_cos']
all_seasons = ['cyc_week_sin', 'cyc_week_cos', 'cyc_month_sin', 'cyc_month_cos', 'cyc_quarter_sin', 'cyc_quarter_cos',
               'cyc_year_sin', 'cyc_year_cos']
slopes = ['Close_slope10', 'Close_slope25', 'Close_slope50']

selected_combos = {
    "h_5": {
        'lag_fast+trend_ratio_slow+momentum_moderate': lag_fast+trend_ratio_slow+momentum_moderate,
        'lag_fast+trend_slow+momentum_moderate': lag_fast+trend_slow+momentum_moderate,
        'duration_slow+trend_slow+momentum_moderate': duration_slow+trend_slow+momentum_moderate,
        'duration_fast+trend_slow+momentum_moderate': duration_fast+trend_slow+momentum_moderate,
        'lag_fast+trend_moderate+trend_ratio_slow': lag_fast+trend_moderate+trend_ratio_slow,
    },
    "h_10": {
        'lag_fast+duration_slow+momentum_moderate': lag_fast+duration_slow+momentum_moderate,
        'lag_fast+duration_fast+momentum_moderate': lag_fast+duration_fast+momentum_moderate,
        'lag_fast+trend_fast+momentum_moderate': lag_fast+trend_fast+momentum_moderate,
        'trend_slow+trend_ratio_slow+trend_ratio_fast': trend_slow+trend_ratio_slow+trend_ratio_fast,
        'trend_slow+trend_ratio_slow+trend_ratio_moderate': trend_slow+trend_ratio_slow+trend_ratio_moderate,
    },
    "h_15": {
        'duration_fast+trend_slow+trend_ratio_slow': duration_fast+trend_slow+trend_ratio_slow,
        'trend_slow+trend_ratio_slow+trend_ratio_moderate': trend_slow+trend_ratio_slow+trend_ratio_moderate,
        'lag_fast+trend_ratio_slow+trend_ratio_moderate': lag_fast+trend_ratio_slow+trend_ratio_moderate,
        'lag_fast+duration_slow+trend_slow': lag_fast+duration_slow+trend_slow,
        'duration_slow+trend_ratio_moderate+trend_ratio_fast': duration_slow+trend_ratio_moderate+trend_ratio_fast,
    },
    "h_20": {
        'trend_slow+trend_ratio_slow+momentum_moderate': trend_slow+trend_ratio_slow+momentum_moderate,
        'lag_fast+duration_slow+trend_ratio_moderate': lag_fast+duration_slow+trend_ratio_moderate,
        'lag_fast+duration_slow+trend_slow': lag_fast+duration_slow+trend_slow,
        'duration_slow+trend_ratio_moderate+volatility_moderate': duration_slow+trend_ratio_moderate+volatility_moderate,
        'lag_fast+trend_ratio_fast+momentum_moderate': lag_fast+trend_ratio_fast+momentum_moderate,
    },
    "h_25": {
        'duration_slow+trend_slow+trend_ratio_moderate': duration_slow+trend_slow+trend_ratio_moderate,
        'duration_slow+duration_fast+trend_ratio_moderate': duration_slow+duration_fast+trend_ratio_moderate,
        'trend_slow+trend_ratio_slow+trend_ratio_moderate': trend_slow+trend_ratio_slow+trend_ratio_moderate,
        'lag_fast+trend_ratio_moderate+volatility_moderate': lag_fast+trend_ratio_moderate+volatility_moderate,
        'duration_slow+trend_moderate+momentum_slow': duration_slow+trend_moderate+momentum_slow,
    },
    "h_35": {
        'duration_slow+duration_moderate+all_seasons': duration_slow+duration_moderate+all_seasons,
        'duration_slow+duration_moderate+momentum_slow': duration_slow+duration_moderate+momentum_slow,
        'duration_slow+momentum_slow+momentum_moderate': duration_slow+momentum_slow+momentum_moderate,
        'duration_slow+duration_fast+momentum_slow': duration_slow+duration_fast+momentum_slow,
        'trend_slow+trend_moderate+trend_ratio_fast': trend_slow+trend_moderate+trend_ratio_fast,
    },
    "h_45": {
        'duration_slow+duration_moderate+momentum_slow': duration_slow+duration_moderate+momentum_slow,
        'duration_slow+momentum_slow+momentum_moderate': duration_slow+momentum_slow+momentum_moderate,
        'duration_slow+duration_moderate+all_seasons': duration_slow+duration_moderate+all_seasons,
        'momentum_moderate+momentum_fast+all_seasons': momentum_moderate+momentum_fast+all_seasons,
        'duration_moderate+momentum_fast+all_seasons': duration_moderate+momentum_fast+all_seasons,
    }
}

def get_model_set(horizon):
    if horizon in [5]:
        key = "h_5"
    elif horizon in [10]:
        key = "h_10"
    elif horizon in [15]:
        key = "h_15"
    elif horizon in [20]:
        key = "h_20" 
    elif horizon in [25]:
        key = "h_25"
    elif horizon in [35]:
        key = "h_35"
    elif horizon in [45]:
        key = "h_45"
    else:
        raise ValueError(f"Unsupported horizon: {horizon}")
    
    return selected_combos[key]

# Velocity x Category Combo Groupings

In [58]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.seterr(invalid='ignore', divide='ignore')
ticker = 'QQQ'
thresh = .5
results = []
r = 5
returns = [5]
arch = 'deep'
lb = 7
test_size = 150
name = 'lag_fast+trend_ratio_slow+momentum_moderate'
cols = lag_fast+trend_ratio_slow+momentum_moderate
opt = 'matthews_corrcoef'

df = extract(ticker, returns, lb, raw_all, windows=[10,25])
df = add_cyclic_seasonality(df, date_col='Date', add_weekly=True, add_month=True,
                            add_quarter=True, add_year=True, add_day_of_year=False,
                            mode='calendar',  
                            prefix='cyc_')    

df_ph = df.copy()
df_ph = df_ph.iloc[r:].copy()

return_col = f"Return_{r}"
return_perc_col = f"Return%_{r}"

rets = df_ph[return_perc_col].copy()
# Split into negative and positive returns
neg = rets[rets < 0]
pos = rets[rets > 0]
# Calculate dynamic thresholds
neg_cutoff = neg.nlargest(int(len(neg) * 0.05)).min()  # least negative of top 10% in magnitude
pos_cutoff = pos.nsmallest(int(len(pos) * 0.05)).max()  # smallest positive of top 10% in magnitude

filtered = df_ph[(df_ph[f'Return%_{r}'] < neg_cutoff) | (df_ph[f'Return%_{r}'] > pos_cutoff)].copy()
counts = filtered[return_col].value_counts()
negf = counts.get(0, 0)
posf = counts.get(1, 1)  # prevent division by zero
#print(f'{posf} | {negf}')
scale_pos_weight = negf / posf

used_cols = cols + [return_col]

df_model = filtered[used_cols].dropna()
df_indicators = df_model[cols]
df_indicators = df_indicators.replace([np.inf, -np.inf], 0)
df_predict = df_model[return_col]

X_test = df_indicators.iloc[:test_size].copy()
y_test = df_predict.iloc[:test_size].copy()

df_indicators = df_indicators.iloc[test_size:].sort_index(ascending=True).copy()
df_predict = df_predict.iloc[test_size:].sort_index(ascending=True).copy()

#X_val = df_indicators.iloc[:val_size].copy()
#y_val = df_predict.iloc[:val_size].copy()
X_train = df_indicators.copy()
y_train = df_predict.copy()

counts = y_train.value_counts()
negf = counts.get(0, 0)
posf = counts.get(1, 1)  # prevent division by zero
#print(f'{posf} | {negf}')
scale_pos_weight = negf / posf
#print(round(scale_pos_weight, 2))

if arch == 'deep':

    # Deep
    xgboost_hyperparameters = {
        'max_depth': [6, 9, 12],          # Still fairly deep
        'learning_rate': [0.01, 0.05],    # Lower rates
        'min_child_weight': [7, 9, 12],   # More conservative splits
        'subsample': [0.7, 0.8, 0.9],     
        'colsample_bytree': [0.75, 0.85], 
        'gamma': [0.3, 0.5],         
        'alpha': [0.1, 0.5, 1.0],         
        'lambda': [5, 10, 15],            
        'n_estimators': [400, 600, 800],
        #'early_stopping_rounds': [50]
    }



In [59]:
thresh = [.5]

def run_cv(cv_split, iter):
    model = XGBClassifier(random_state=42)
    model.set_params(n_jobs=1)
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=xgboost_hyperparameters,
        scoring=opt,
        cv=cv_split,
        n_jobs=-1,
        n_iter=iter,
        random_state=42
    )
    
    random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    #random_search.fit(X_train, y_train, verbose=False)
    best_model = random_search.best_estimator_
    y_prob = best_model.predict_proba(X_test)
    postot = y_test.sum()
    negtot = len(y_test) - y_test.sum()

    metrics = {}
    for t in thresh:
        y_pred_thresh = (y_prob[:, 1] > t).astype(int)
        y_pred_thresh[y_prob[:, 0] > t] = 0
        filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

        if filtered_indices.sum() > 0:
            y_test_valid = y_test[filtered_indices]
            y_pred_valid = y_pred_thresh[filtered_indices]
            posprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 2)
            negprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 2)
            poscnt = sum(y_pred_valid == 1)
            negcnt = sum(y_pred_valid == 0)
            posrec = round(posprec * poscnt / postot, 2)
            negrec = round(negprec * negcnt / negtot, 2)
            metrics[t] = {
            "Len": len(y_test),
            "PC": poscnt,
            "NC": negcnt,
            "PP": posprec,
            "PR": posrec,
            "NP": negprec,
            "NR": negrec,
            "PosFb": round((3*posprec*posrec)/((2*posrec)+(1*posprec)),3),
            "NegFb": round((3*negprec*negrec)/((2*negrec)+(1*negprec)),3),
        }
    return metrics, best_model

xgboost_hyperparameters = {
    'max_depth': [6, 9, 12],          # Still fairly deep
    'learning_rate': [0.01, 0.05],    # Lower rates
    'min_child_weight': [4, 7, 9, 12],   # More conservative splits
    'subsample': [0.7, 0.8, 0.9],     
    'colsample_bytree': [0.75, 0.85], 
    'colsample_bylevel': [0.7, 0.8, 0.9],
    'colsample_bynode': [0.7, 0.8, 0.9],    
    'gamma': [0.1, 0.3, 0.5],         
    'alpha': [0.1, 0.5, 1.0],         
    'lambda': [5, 10, 15],            
    'n_estimators': [400, 600, 800, 1000], 
    'early_stopping_rounds': [10]
    }

iter = 50

# Try all split values and pick best
best_result = None
best_score = -float("inf")

split_values = [3, 4, 5]

for splits in split_values:

    cv_split = TimeSeriesSplit(n_splits=splits)

    metrics, model = run_cv(cv_split, iter)

    # scoring method: average of PosFb and NegFb at first threshold
    t = list(metrics.keys())[0]
    avg_score = (metrics[t]["PosFb"] + metrics[t]["NegFb"]) / 2

    if avg_score > best_score:
        best_score = avg_score
        best_result = (metrics, model, splits)

print(f"Best CV split: {best_result[2]} (score={best_score.round(2)})")
    

Best CV split: 4 (score=0.54)


In [65]:
splits = 4
cv_split = TimeSeriesSplit(n_splits=splits)
model = XGBClassifier(random_state=42)

model.set_params(n_jobs=1)
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=xgboost_hyperparameters,
    scoring=opt,
    cv=cv_split,
    n_jobs=-1,
    n_iter=iter,
    random_state=42
)

random_search.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
best_model = random_search.best_estimator_
y_prob = best_model.predict_proba(X_test)
postot = y_test.sum()
negtot = len(y_test) - y_test.sum()

metrics = {}
thresh = [.5]
for t in thresh:
    y_pred_thresh = (y_prob[:, 1] > t).astype(int)
    y_pred_thresh[y_prob[:, 0] > t] = 0
    filtered_indices = (y_prob[:, 1] > t) | (y_prob[:, 0] > t)

    if filtered_indices.sum() > 0:
        y_test_valid = y_test[filtered_indices]
        y_pred_valid = y_pred_thresh[filtered_indices]
        posprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=1, zero_division=0), 2)
        negprec = round(precision_score(y_test_valid, y_pred_valid, pos_label=0, zero_division=0), 2)
        poscnt = sum(y_pred_valid == 1)
        negcnt = sum(y_pred_valid == 0)
        posrec = round(posprec * poscnt / postot, 2)
        negrec = round(negprec * negcnt / negtot, 2)
        metrics[t] = {
        "Len": len(y_test),
        "PC": (poscnt * posprec / posrec).astype(int),
        "NC": (negcnt * negprec / negrec).astype(int),
        "PP": posprec,
        "PR": posrec,
        "NP": negprec,
        "NR": negrec,
        "PosFb": round((3*posprec*posrec)/((2*posrec)+(1*posprec)),3),
        "NegFb": round((3*negprec*negrec)/((2*negrec)+(1*negprec)),3),
    }


In [67]:
best_model

XGBClassifier(alpha=0.1, base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.75, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=0.5, grow_policy=None, importance_type=None,
              interaction_constraints=None, lambda=5, learning_rate=0.05,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=12, max_leaves=None,
              min_child_weight=12, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=800, n_jobs=1, ...)

In [169]:
                best_xg_model, score = optimize_splits_new(df_indicators, df_predict, thresh, opt, arch, test_size, withold, return_metrics=False)

                if score > best_score:
                    best_score = score
                    best_model = best_xg_model
                    best_lb = lb

            if best_model is not None:
                print(f"Saving best model: lb={best_lb}, score={round(best_score,3)}")
                #savers(best_model, r, name, ticker, arch)

                #print(f"Results for pi adjusted {name} | {arch} | {ticker}_{r}")
                perm_size = test_size
                X_test = df_indicators.iloc[:perm_size].copy()
                y_test = df_predict.iloc[:perm_size].copy()
                
                # Run permutation importance
                perm = permutation_importance(best_model, X_test, y_test, n_repeats=100, random_state=None, n_jobs=-1)

                # Organize into a DataFrame
                perm_df = pd.DataFrame({
                    'feature': df_indicators.columns,
                    'importance_mean': perm.importances_mean,
                    'importance_std': perm.importances_std,
                    'arch': arch,
                    'horizon': r,
                    'model_name': name
                })

                all_perm_dfs.append(perm_df)
        
                print(f"Permutation Importance — {name} | QQQ_{r}")
            
            print('---------------------------')

17 | 17
Results for 8lb | lag_fast+trend_ratio_slow+momentum_moderate | deep | QQQ_5
Best CV split: 5 (score=0.65)
  Threshold 0.5: {'Len': 150, 'PC': 128, 'NC': 22, 'PP': 0.73, 'PR': 0.94, 'NP': 0.73, 'NR': 0.31, 'PosFb': 0.789, 'NegFb': 0.503}
17 | 17
Results for 10lb | lag_fast+trend_ratio_slow+momentum_moderate | deep | QQQ_5
Best CV split: 3 (score=0.64)
  Threshold 0.5: {'Len': 150, 'PC': 120, 'NC': 30, 'PP': 0.73, 'PR': 0.88, 'NP': 0.63, 'NR': 0.37, 'PosFb': 0.774, 'NegFb': 0.51}
Saving best model: lb=8, score=0.646
Permutation Importance — lag_fast+trend_ratio_slow+momentum_moderate | QQQ_5
---------------------------


Exception ignored from cffi callback <function buffer_callback at 0x110db98a0>:
Traceback (most recent call last):
  File "/Users/brettchase/Library/Python/3.12/lib/python/site-packages/curl_cffi/curl.py", line 63, in buffer_callback
    @ffi.def_extern()
    
KeyboardInterrupt: 


26 | 26
Results for 8lb | lag_fast+trend_slow+momentum_moderate | deep | QQQ_5
Best CV split: 4 (score=0.66)
  Threshold 0.5: {'Len': 150, 'PC': 107, 'NC': 43, 'PP': 0.76, 'PR': 0.82, 'NP': 0.58, 'NR': 0.49, 'PosFb': 0.779, 'NegFb': 0.547}
26 | 26
Results for 10lb | lag_fast+trend_slow+momentum_moderate | deep | QQQ_5


KeyboardInterrupt: 

In [139]:
# Combine all permutation importance DataFrames
#current_perm_df = pd.concat(all_perm_dfs, ignore_index=True)

# Apply the retention logic per group
def apply_retention_logic(group):
    group = group.copy()
    group["drop_flag"] = group["importance_mean"] <= 0.01

    total = len(group)
    retained = (~group["drop_flag"]).sum()
    min_retained = int(np.ceil(total * 0.45))

    if retained < min_retained:
        top_feats = group.sort_values(by="importance_mean", ascending=False).head(min_retained)["feature"]
        group["drop_flag"] = ~group["feature"].isin(top_feats)

    return group

# Apply group-wise adjustment by model_name, arch, horizon
current_perm_df = current_perm_df.groupby(
    ["model_name", "arch", "horizon"], group_keys=False
).apply(apply_retention_logic)

# Build feature map from retained features
curr_feature_map = (
    current_perm_df[~current_perm_df["drop_flag"]]
    .groupby(["model_name", "arch", "horizon"])["feature"]
    .apply(list)
    .to_dict()
)

# Save
with open("curr_feature_map_v2.pkl", "wb") as f:
    pickle.dump(curr_feature_map, f)

/var/folders/k0/mlnk5_mx6ns64dfxsknt0p3m0000gn/T/ipykernel_31053/314718624.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(apply_retention_logic)


In [148]:
ticker = 'QQQ'
model_dir = f'../Models/Ensemble_{ticker}'
thresh = .5
arch_types = ['deep']
lb = 5

with open("curr_feature_map_v2.pkl", "rb") as f:
        feature_map = pickle.load(f)

def submodel(thresh):

        upper = thresh
        lower = 1 - upper

        # Placeholder for final predictions
        combined_predictions = {}
        df = extract(ticker, returns, lb, raw_all, windows=[10,25])
        df = add_cyclic_seasonality(df, 
                                    date_col='Date',
                                    add_weekly=True,
                                    add_month=True,
                                    add_quarter=True,
                                    add_year=True,
                                    add_day_of_year=False,
                                    mode='calendar',    # or 'trading' or 'both'
                                    prefix='cyc_')

        # Iterate through each return horizon
        for r in returns:

            return_col = f"Return_{r}"

            df_ph = df.copy()

            # Create base prediction matrix with Date and Close
            pred_matrix = df_ph[['Date', 'Close']].copy()

            for arch in arch_types:
            
                selected_models = get_model_set(r)
                    
                # all index-combos of length r
                for name, original_cols in selected_models.items():
                    
                    cols = feature_map.get((name, arch, r), original_cols)
                    df_features = df_ph[cols].copy()
                    df_features = df_features.replace([np.inf, -np.inf], 0)

                    # XGBoost prediction
                    xgb_path = os.path.join(model_dir, f"{name}_xgboost_{r}{arch}rs.pkl")
                    if os.path.exists(xgb_path):
                        with open(xgb_path, 'rb') as f:
                            model_xgb = pickle.load(f)
                        prob_xgb = model_xgb.predict_proba(df_features)[:, 1]
                        col_name_xgb = f"{name}_xgb_{r}{arch}"
                        pred_matrix[col_name_xgb] = np.where(
                            prob_xgb > upper, prob_xgb,
                            np.where(prob_xgb < lower, prob_xgb - 1, 0)
                        )

                        signed_pred = np.where(
                            prob_xgb > upper, prob_xgb,
                            np.where(prob_xgb < lower, prob_xgb - 1, 0)
                        )


                        """
                        # scalar weights for this model
                        pw = pos_w_map.get(col_name_xgb)
                        nw = neg_w_map.get(col_name_xgb)

                        weighted_col_name_xgb = f"weighted_{name}_xgb_{r}{arch}"
                        weights = np.where(signed_pred > 0, pw, np.where(signed_pred < 0, -nw, 0))
                        pred_matrix[weighted_col_name_xgb] = weights * signed_pred
                        """

                    # Final summed prediction across models
                    pred_cols = [col for col in pred_matrix.columns if col not in ['Date', 'Close'] and not col.startswith('sum_')
                                 and not col.startswith('w_sum_') and not col.startswith('weighted_')]
                    pred_matrix[f"sum_{r}"] = pred_matrix[pred_cols].sum(axis=1)
                    #weighted_cols = [col for col in pred_matrix.columns if col not in ['Date', 'Close'] and not col.startswith('sum_')
                    #             and not col.startswith('w_sum_') and col.startswith('weighted_')]
                    #pred_matrix[f"w_sum_{r}"] = pred_matrix[weighted_cols].sum(axis=1)

                    combined_predictions[r] = pred_matrix

        aggr_df = None

        for r, df_preds in combined_predictions.items():
            cols_to_merge = [col for col in df_preds.columns if col not in ['Date', 'Close']]
            df_preds_renamed = df_preds[['Date', 'Close'] + cols_to_merge].copy()

            if aggr_df is None:
                aggr_df = df_preds_renamed
            else:
                aggr_df = pd.merge(aggr_df, df_preds_renamed, on=['Date', 'Close'], how='outer')

        sub_model_df = aggr_df.copy()

        return sub_model_df

sub_model_df = submodel(thresh)
sub_model_df[['sum_5', 'sum_10', 'sum_15', 'sum_20', 'sum_25', 'sum_35',
              'sum_45']].sort_index(ascending=False).head(30).round(3).reset_index()

5
5
5
5
5


ValueError: feature_names mismatch: ['OBV_ROC5', 'Min_60_Rows_Since', 'Min_120_Rows_Since', 'num_days_100', 'num_days_200', 'ADL', 'CMF_10', 'CCI_14', 'OBV'] ['Min_60_Rows_Since', 'Min_120_Rows_Since', 'num_days_100', 'num_days_200', 'ADL', 'CCI_14', 'OBV']
expected OBV_ROC5, CMF_10 in input data

In [23]:
pos_w_map = accuracy_df.groupby("model")["pos_weight"].mean().to_dict()
neg_w_map = accuracy_df.groupby("model")["neg_weight"].mean().to_dict()

KeyError: 'model'

In [ ]:
# BEFORE any loops:
feature_importance_records = []

for r in returns:
    return_col = f"Return_{r}"
    df_ph = df.copy()
    pred_matrix = df_ph[['Date','Close']].copy()

    for arch in arch_types:
        selected_models = get_model_set(r)
        
        for name, cols in selected_models.items():
            # prepare features
            df_features = df_ph[cols].replace([np.inf, -np.inf], 0)

            # load model if it exists
            xgb_path = os.path.join(model_dir, f"{name}_xgboost_{r}.pkl")
            if not os.path.exists(xgb_path):
                continue
            with open(xgb_path,'rb') as f:
                model_xgb = pickle.load(f)

            # 1) record the top-5 importances
            importances = model_xgb.feature_importances_
            feat_imp_pairs = list(zip(cols, importances))
            for feature, imp in sorted(feat_imp_pairs, key=lambda x: x[1], reverse=True)[:5]:
                feature_importance_records.append({
                    'horizon':   r,
                    'arch':      arch,
                    'model':     name,
                    'feature':   feature,
                    'importance': imp
                })

            # 2) your existing prediction code
            prob_xgb     = model_xgb.predict_proba(df_features)[:,1]
            col_name_xgb = f"{name}_xgb_{r}{arch}"
            pred_matrix[col_name_xgb] = np.where(
                prob_xgb > upper, prob_xgb,
                np.where(prob_xgb < lower, prob_xgb - 1, 0)
            )

# AFTER all loops:
fi_df = pd.DataFrame(feature_importance_records)

In [ ]:
import pandas as pd

# never wrap a wide DataFrame across multiple lines
pd.set_option('display.expand_frame_repr', False)
# show all columns
pd.set_option('display.max_columns', None)
# pretend the display is super wide
pd.set_option('display.width', 10000)
       

In [24]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    f1_score, accuracy_score
)
threshs = [.55]
length = 130
accuracy_df = pd.DataFrame()

def performance_metrics(metrics_df, returns):

    metrics_df = metrics_df.sort_index(ascending=True)

    def add_column_based_on_future_value(df, days):

        future_return = (df['Close'].shift(-days) - df['Close']) / df['Close']

        if days >= 100:
            df[f'Return_{days}'] = np.where(
                future_return > 0.001, 1,
                np.where(future_return < -0.001, 0, np.nan)
            )
        else:
            df[f'Return_{days}'] = (future_return > 0).astype(int)

        return df

    # Apply return logic for each target horizon
    for r in returns: 
        new_df = add_column_based_on_future_value(metrics_df, r)

    return new_df

for thresh in threshs:
    
    sub_model_df = submodel(thresh)
    performance_df = performance_metrics(sub_model_df.round(3), returns)

    horizons = returns
    # Dictionary to store the filtered DataFrames
    horizon_dfs = {}

    # Loop through each horizon
    for h in horizons:
        # Collect relevant columns
        cols = ['Date', 'Close'] + [
            col for col in performance_df.columns 
            if col.endswith(f'_xgb_{h}deep') or col.endswith(f'_xgb_{h}shallow') or col == f'sum_{h}' or col == f'Return_{h}'
        ]
        
        # Create and store the filtered DataFrame
        horizon_dfs[h] = performance_df[cols].copy()

    def convert_signed_to_prob(p):
        return p if p >= 0 else 1 + p

    for h in horizons:

        df = horizon_dfs[h].sort_values('Date', ascending=False).iloc[h:h+length, :].copy()
        
        for arch in ['deep']:
            records = []
            actual_pos = df[f'Return_{h}'].sum()
            actual_neg = len(df[f'Return_{h}']) - actual_pos
            for col in df.columns:
                if f'_xgb_{h}{arch}' in col:
                    # only keep the rows that actually got a prediction
                    filtered = df[df[col] != 0]
                    # convert your raw score to a probability
                    probs = filtered[col].apply(convert_signed_to_prob)
                    # binarize at 0.5
                    y_pred = (probs >= 0.5).astype(int)
                    y_true = filtered[f'Return_{h}']
                    
                    # compute the matrix
                    cm = confusion_matrix(y_true, y_pred,labels=[0,1])
                    tn, fp, fn, tp = cm.ravel()
                    records.append({
                        'thresh': thresh,
                        'model': col,
                        'act_pos': actual_pos,
                        'act_neg': actual_neg,
                        'pos_acc': f"{(tp/(tp + fp)):.1%}" if actual_pos>0 else "n/a",
                        'neg_acc': f"{(tn/(tn + fn)):.1%}" if actual_neg>0 else "n/a",
                        'tp': tp,
                        'fp': fp,
                        'tn': tn,
                        'fn': fn,
                    })

            # make your summary table
            cm_df = pd.DataFrame(records)
            accuracy_df = pd.concat([accuracy_df, cm_df], ignore_index=True)

accuracy_df["pos_weight"] = (accuracy_df["pos_acc"].str.rstrip("%").astype(float).fillna(33).clip(lower=33) / 100)
accuracy_df["neg_weight"] = (accuracy_df["neg_acc"].str.rstrip("%").astype(float).fillna(33).clip(lower=33) / 100)
accuracy_df

NameError: name 'pos_w_map' is not defined

In [25]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_curve, auc,
    precision_recall_curve, average_precision_score,
    f1_score, accuracy_score
)

def performance_metrics(metrics_df, returns):

    metrics_df = metrics_df.sort_index(ascending=True)

    def add_column_based_on_future_value(df, days):

        future_return = (df['Close'].shift(-days) - df['Close']) / df['Close']

        if days >= 100:
            df[f'Return_{days}'] = np.where(
                future_return > 0.001, 1,
                np.where(future_return < -0.001, 0, np.nan)
            )
        else:
            df[f'Return_{days}'] = (future_return > 0).astype(int)

        return df

    # Apply return logic for each target horizon
    for r in returns: 
        new_df = add_column_based_on_future_value(metrics_df, r)

    return new_df

performance_df = performance_metrics(sub_model_df.round(3), returns)

horizons = returns
# Dictionary to store the filtered DataFrames
horizon_dfs = {}

# Loop through each horizon
for h in horizons:
    # Collect relevant columns
    cols = ['Date', 'Close'] + [
        col for col in performance_df.columns 
        if col.endswith(f'_xgb_{h}deep') or col.endswith(f'_xgb_{h}shallow') or col == f'sum_{h}' or col == f'Return_{h}'
    ]
    
    # Create and store the filtered DataFrame
    horizon_dfs[h] = performance_df[cols].copy()

def convert_signed_to_prob(p):
    return p if p >= 0 else 1 + p

for h in horizons:
    df = horizon_dfs[h].sort_values('Date', ascending=False).iloc[h:h+50, :].copy()
    
    plt.figure(figsize=(10, 6))
    for col in df.columns:
        if f'_xgb_{h}' in col: #sum_{h}
            filtered = df[df[col] != 0]
            probs = filtered[col].apply(convert_signed_to_prob)
            targets = filtered[f'Return_{h}']
            fpr, tpr, _ = roc_curve(targets, probs)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f'{col} (AUC = {roc_auc:.2f})')

    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'ROC Curve for All Models - Horizon {h}')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.grid()
    plt.show()
    
    # --- Calibration Curve ---
    plt.figure(figsize=(10, 6))
    for col in df.columns:
        if f'_xgb_{h}' in col:
            filtered = df[df[col] != 0]
            probs = filtered[col].apply(convert_signed_to_prob)
            targets = filtered[f'Return_{h}']
            prob_true, prob_pred = calibration_curve(targets, probs, n_bins=50)
            plt.plot(prob_pred, prob_true, marker='o', label=col)

    plt.plot([0, 1], [0, 1], 'k--')
    plt.title(f'Calibration Curve - Horizon {h}')
    plt.xlabel('Predicted Probability')
    plt.ylabel('Actual Frequency')
    plt.legend()
    plt.grid()
    plt.show()
    

    for arch in ['deep']:
        records = []
        actual_pos = df[f'Return_{h}'].sum()
        actual_neg = len(df[f'Return_{h}']) - actual_pos
        for col in df.columns:
            if f'_xgb_{h}{arch}' in col:
                # only keep the rows that actually got a prediction
                filtered = df[df[col] != 0]
                # convert your raw score to a probability
                probs = filtered[col].apply(convert_signed_to_prob)
                # binarize at 0.5
                y_pred = (probs >= 0.5).astype(int)
                y_true = filtered[f'Return_{h}']
                
                # compute the matrix
                cm = confusion_matrix(y_true, y_pred,labels=[0,1])
                tn, fp, fn, tp = cm.ravel()
                records.append({
                    'model': col,
                    'act_pos': actual_pos,
                    'act_neg': actual_neg,
                    'pos_acc': f"{(tp/(tp + fp)):.1%}" if actual_pos>0 else "n/a",
                    'neg_acc': f"{(tn/(tn + fn)):.1%}" if actual_neg>0 else "n/a",
                    'tp': tp,
                    'fp': fp,
                    'tn': tn,
                    'fn': fn,
                })

        # make your summary table
        cm_df = pd.DataFrame(records)
        print(cm_df.to_string())
        print(' ')

NameError: name 'sub_model_df' is not defined